In [1]:
import os
import shutil

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

tqdm==4.64.1
numpy==1.23.4
pandas==1.2.4
scikit_learn==0.24.1
boto3==1.24.59

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import boto3
import pickle

# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

# constants
str_project = '20231010-gen-xii'
str_datecol = 'applicationdate__app'
str_target = 'target'
str_dirname_output = './output'
str_model = '02_pricing_pd'

# create output dir
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

# get module
print('Downloading module for preprocessing...')
str_filename = 'preprocessing.py'
str_local_path = f'./{str_filename}'
str_bucket_path = f'01_ad/02_model/00_preprocessing/01_create_preprocessor/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

# import preprocessing model
print('Importing preprocessing model...')
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'01_ad/02_model/00_preprocessing/01_create_preprocessor/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))

# preprocess and save data
for str_df in ['train','valid','test']:
    # import data
    print(f'Importing {str_df} data...')
    str_filename = f'df_{str_df}_noleaks.gzip'
    str_uri = f's3://{str_project}/{str_model}/01_data_prep/05_leaky_features/04_write_dfs/{str_filename}'
    df = pd.read_parquet(str_uri)
    list_target = list(df[str_target])
    # drop target so it doesnt get preprocessed
    df.drop(str_target, axis=1, inplace=True)
    print('')

    # preprocess
    print(f'Preprocessing {str_df} data...')
    df = cls_model_preprocessing.transform(df)
    # re-assign target now that the data has been preprocessed
    df[str_target] = list_target
    print('')
    
    # get non-numeric
    list_non_numeric = []
    for col in df.columns:
        if df[col].dtype not in ['float64','int64']:
            list_non_numeric.append(col)
    # rm date col
    list_non_numeric = [col for col in list_non_numeric if col != str_datecol]
    
    # non-numeric to string
    print('Converting non-numeric to string...')
    df[list_non_numeric] = df[list_non_numeric].astype(str)
    
    # write full data set to s3
    print(f'Writing {str_df} data to s3...')
    str_filename = f'df_{str_df}_noleaks_pre.gzip'
    str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
    df.to_parquet(str_uri, compression='gzip')
    
    # logic for writing samples of training data
    if str_df == 'train':
        # write sample to s3
        print('Writing 75% sample to s3...')
        df = df.sample(frac=0.75, random_state=42) # 75% of 100%
        str_filename = f'df_{str_df}_noleaks_pre_75.gzip'
        str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
        df.to_parquet(str_uri, compression='gzip')
        print('')
        
        # write sample to s3
        print('Writing 50% sample to s3...')
        df = df.sample(frac=0.666, random_state=42) # 66% of 75% is 50% of 100%
        str_filename = f'df_{str_df}_noleaks_pre_50.gzip'
        str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
        df.to_parquet(str_uri, compression='gzip')
        print('')
        
        # write sample to s3
        print('Writing 25% sample to s3...')
        df = df.sample(frac=0.50, random_state=42) # 50% of 66.6% is 25% of 100%
        str_filename = f'df_{str_df}_noleaks_pre_25.gzip'
        str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
        df.to_parquet(str_uri, compression='gzip')
        print('')
        
        # write sample to s3
        print('Writing 1% sample to s3...')
        df = df.sample(frac=0.04, random_state=42) # 4% of 25% is 1% of 100%
        str_filename = f'df_{str_df}_noleaks_pre_1.gzip'
        str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
        df.to_parquet(str_uri, compression='gzip')
    else:
        pass

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxii-pd-make-dfs

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  23.04kB
Step 1/7 : FROM python:3.9
 ---> 7ef94ac333fa
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> 529128453704
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 8f56313abc92
Step 4/7 : COPY requirements.txt .
 ---> 976262b82a0b
Step 5/7 : RUN pip install -r requirements.txt
 ---> Running in 583eb42931ba
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 89.6 MB/s eta 0:0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.3/304.3 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 9.1 MB/s eta 0:00:00
Removing intermediate container 583eb42931ba
 ---> 74a6aaa8295f
Step 6/7 : COPY script.py .
 ---> 8318a7e48e27
Step 7/7 : CMD ["python3", "script.py"]
 ---> Running in 379e4b57ad2b
Removing intermediate container 379e4b57ad2b
 ---> 42b100485e86
Successfully built 42b100485e86
Successfully tagged genxii-pd-make-dfs:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-pd-make-dfs' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-make-dfs]
89935a71c3d4: Preparing
98d39146b6c1: Preparing
b2ded54cfdc2: Preparing
7f7a9ee63288: Preparing
781f058a9424: Preparing
78ecb2a2f011: Preparing
84062ebc4cf5: Preparing
2180aea5f54b: Preparing
86388e04a96b: Preparing
893507f6057f: Preparing
2353f7120e0e: Preparing
51a9318e6edf: Preparing
c5bb35826823: Preparing
86388e04a96b: Waiting
84062ebc4cf5: Waiting
2180aea5f54b: Waiting
893507f6057f: Waiting
2353f7120e0e: Waiting
51a9318e6edf: Waiting
c5bb35826823: Waiting
78ecb2a2f011: Waiting
b2ded54cfdc2: Pushed
89935a71c3d4: Pushed
84062ebc4cf5: Pushed
78ecb2a2f011: Pushed
7f7a9ee63288: Pushed
781f058a9424: Pushed
86388e04a96b: Pushed
2180aea5f54b: Pushed
51a9318e6edf: Pushed
c5bb35826823: Pushed
2353f7120e0e: Pushed
893507f6057f: Pushed
98d39146b6c1: Pushed
latest: digest: sha256:2fcba84dfe897b897dc3f9aa491cc01e2066cab419a4ae830295fdd006497f50 size: 3058


### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass